In [ ]:
import os
import pandas as pd
import torch
import numpy as np
import matplotlib.pyplot as plt

from utils import loadexp
import metrics

### Setup (modify only the cell in this section)

In [ ]:
# To load data
datapath = os.path.join('..', 'data')
exp_rod = (
    os.path.join(datapath, 'rod_mesh.xml'), 
    os.path.join(datapath,'rod_snapshots.npz')
)
exp_shield = (
    os.path.join(datapath, 'shield_mesh.xml'), 
    os.path.join(datapath,'shield_snapshots.npz')
)
exp_gaussian = (
    os.path.join(datapath, 'gaussian_mesh.xml'), 
    os.path.join(datapath,'gaussian_snapshots.npz')
)

# To load raytune analysis
analysespath = os.path.join('..', 'results', 'analyses')

# Loads raytune analysis given the experiment name
def get_tc(name):
    if name == 'pga':
        exp_name = os.path.join(analysespath, 'hypexp_pga.obj')
        (utrain, uval, utest), _ = loadexp(*exp_gaussian, split = [200, 100, 0])
    elif name == 'rod':
        exp_name = os.path.join(analysespath, 'hypexp_rod.obj')
        (utrain, uval, utest), _ = loadexp(*exp_rod, split = [200, 100, 0])
    elif name == 'els':
        exp_name = os.path.join(analysespath, 'hypexp_els.obj')
        (utrain, uval, utest), _ = loadexp(*exp_shield, split = [200, 100, 0])
    else:
        raise ValueError()
    return exp_name, (utrain, uval, utest)

### Functions to extract data from analysis

In [ ]:
# Select only relevant columns
def get_df_with_selected_columns(
    exp_name : str, 
    display_bool : bool = False
):

    # Loads analysis
    result_grid = np.load(exp_name, allow_pickle = True)

    # Select columns
    results_df = result_grid.get_dataframe()
    results_df_selected = results_df.sort_values(by = 'loss')[
        ['loss', 
        'config/lr_cfg', 
        'config/act_cfg',
        'config/red_dims_cfg', 
        'config/init_cfg', 
        'config/scheduler_cfg', 
        'config/seed',
        'time_total_s']
    ]

    # To display table
    if display_bool:
        with pd.option_context('display.max_rows', 2000):
            display(results_df_selected)

    return results_df_selected


# Extract info from dataframe to generate the plots
def get_info_to_plot(results_df_selected : pd.DataFrame):

    # Setup
    nottaken = np.array([True]*len(results_df_selected))
    info_dict = dict()
    info_dict['eys'] = list()
    info_dict['standard'] = list()
    info_dict['latent_dims'] = list()
    info_dict['scheduler'] = list()
    info_dict['lr'] = list()
    cfgs = (
        'config/lr_cfg', 
        'config/red_dims_cfg', 
        'config/scheduler_cfg', 
        'config/act_cfg'
    )

    # Pairs standard and EYS-initialized architectures having the same cfg
    while (sum(nottaken) > 0):

        # Pairing
        first = results_df_selected[nottaken].iloc[0]
        cond = np.ones_like(nottaken)
        for key in cfgs:
            cond *= (results_df_selected[key] == first[key])
        nottaken[cond] = False
        is_eys = results_df_selected[cond]['config/init_cfg'] == 'eys'
        is_standard = results_df_selected[cond]['config/init_cfg'] == 'standard'

        # Get best val MSE loss of {standard, eys} among the runs
        eys = np.array(results_df_selected[cond][is_eys]['loss']).min()
        standard = np.array(results_df_selected[cond][is_standard]['loss']).min()

        # Collects other relevant info
        helper_column = np.array(
            results_df_selected[cond][is_eys]['config/red_dims_cfg']
        )
        curr_sched = results_df_selected[cond][is_eys]['config/scheduler_cfg']
        curr_lr = results_df_selected[cond][is_eys]['config/lr_cfg']
        curr_latent_dim = eval(helper_column[0])[-1]
        info_dict['eys'].append(eys)
        info_dict['standard'].append(standard)
        info_dict['latent_dims'].append(curr_latent_dim)
        info_dict['scheduler'].append(curr_sched.values[0])
        info_dict['lr'].append(curr_lr.values[0])

    # converts each column to array
    for key in info_dict.keys():
        info_dict[key] = np.array(info_dict[key])

    return info_dict


# To compute the POD error
def get_pod_error(
    info_dict : dict, 
    utrain : torch.tensor, 
    uval : torch.tensor
):
    bias = utrain.mean(axis = 0)
    V = torch.linalg.svd((utrain - bias).T)[0]
    Nrange = np.unique(info_dict['latent_dims'])
    all_err_pod = np.zeros((info_dict['latent_dims'].shape[0],))

    for N in Nrange:
        VN = V[:,:N]
        uproj = (uval - bias) @ VN @ VN.T + bias
        curr_err_pod = metrics.mse(utrue = uval, upred = uproj).item()
        all_err_pod[(info_dict['latent_dims'] == N)] = curr_err_pod
   
    info_dict['POD'] = all_err_pod

    return info_dict

### Functions to generate figure

In [ ]:
# Get global minimum and maximum to set xlim and ylim of the plots
def get_min_max_plots(info_dict):
    vmin = np.inf
    vmax = -np.inf
    for key in ('eys', 'standard', 'POD'):
        vmin = min(vmin, info_dict[key].min())
        vmax = max(vmax, info_dict[key].max())
    return vmin, vmax


# To generate the figure
def plot_fig(info_dict, test_case):

    # setup figure and get minmax for plotting
    _, axs = plt.subplots(1, 3, figsize = (12,2.8))
    vmin, vmax = get_min_max_plots(info_dict)

    # Function to plot with
    def plot_one(ax, key1, key2, color):
        ax.loglog(info_dict[key1], info_dict[key2], '.', color = color)
        ax.loglog([vmin, vmax], [vmin, vmax], '-k')
        ax.set_title(test_case.upper() + ': rate (x < y) = %1.2f' %\
            (info_dict[key1] < info_dict[key2]).mean()
        )
        ax.grid(
            visible = True, 
            which = 'major', 
            axis = 'both', 
            linestyle = ':', 
            linewidth = 0.5
        )
        ax.set_xlabel('$MSE_{val}$ w/ %s' % key1.upper(), fontsize = 12)
        ax.set_ylabel('$MSE_{val}$ w/ %s' % key2.upper(), fontsize = 12)

    # plot and adjust
    plot_one(ax = axs[0], key1 = 'eys', key2 = 'standard', color = 'r')
    plot_one(ax = axs[1], key1 = 'eys', key2 = 'POD', color = 'b')
    plot_one(ax = axs[2], key1 = 'standard', key2 = 'POD', color = 'g')
    plt.subplots_adjust(wspace = 0.6)
    
    savepath = os.path.join('..', 'results', 'figures')
    if not os.path.exists(savepath):
        os.makedirs(savepath)
    plt.savefig(
        os.path.join(savepath, 'hypopt_' + test_case + '.png'), 
        bbox_inches = 'tight', 
        dpi = 300
    )

### Figures generation

In [ ]:
for test_case in ('pga', 'rod', 'els'):
    exp_name, (utrain, uval, utest) = get_tc(name = test_case)
    results_df_selected = get_df_with_selected_columns(exp_name)
    info_dict = get_info_to_plot(results_df_selected)
    info_dict = get_pod_error(info_dict, utrain, uval)
    plot_fig(info_dict, test_case)